# B02 — Data Quality and Governance

**Time: about 60 minutes.**
Covers: Managing Data Quality Governance

### What you will be able to do afterwards

- Express quality rules once and use the same definition to filter and to quarantine.
- Route bad rows somewhere diagnosable instead of dropping them.
- Trend quality over time rather than checking it once.
- Enforce at the table level what your code already checks.
- Mask sensitive columns based on who is asking.

### The principle

A pipeline that silently drops 3% of rows is worse than one that fails, because nobody
finds out for six months. Everything in this notebook exists to make bad data **visible**
rather than absent.

In [ ]:
from pyspark.sql import DataFrame, functions as F
from helpers import utils

cfg = utils.get_configs()
catalog = cfg["catalog"]
silver_schema, gold_schema = cfg["schema_silver"], cfg["schema_gold"]

sales_silver = utils.get_configs("sales")["table_silver"]
validated_table = utils.get_configs("sales_validated")["table_silver"]
quarantine_table = utils.get_configs("sales_quarantine")["table_silver"]
metrics_table = utils.get_configs("dq_metrics")["table_gold"]

spark = utils.spark
print(sales_silver, validated_table, quarantine_table, metrics_table, sep="\n")

## Step 1 — Define the rules once

**TO DO**

Define your rules as **data**, not as scattered `WHERE` clauses. A dictionary of
`rule_name -> SQL boolean expression that a good row satisfies` is enough:

```python
RULES = {
    "price_positive":      "price > 0",
    "quantity_positive":   "quantity > 0",
    "product_id_present":  "product_id IS NOT NULL",
    "user_id_present":     "user_id IS NOT NULL",
}
```

Add at least one more rule of your own that the data might plausibly violate.

Then write `evaluate_rules(df, rules)` returning the DataFrame with one boolean column per
rule, plus a `dq_failed_rules` array holding the names of the rules each row failed.

**Why one definition.** If the filter says `price > 0` and the quarantine predicate says
`price >= 0`, rows fall through the gap and are lost. Deriving both from one dictionary
makes that class of bug impossible.

In [ ]:
RULES = {
    # TO DO: define your rules
}


def evaluate_rules(df: DataFrame, rules: dict) -> DataFrame:
    """
    Add one boolean column per rule plus `dq_failed_rules` (array of failed rule names).

    Args:
        df: the DataFrame to check.
        rules: {rule_name: sql_expression_true_when_row_is_good}
    Returns:
        The same rows, annotated. Nothing is filtered here.
    """
    # TO DO
    pass


# TO DO: apply it to the silver sales table and look at the annotated output

## Step 2 — Split, don't drop

**TO DO**

1. Write the rows that pass every rule to `sales_validated` in your silver schema.
2. Write the rows that fail at least one to `sales_quarantine`, keeping:
   - every original column,
   - `dq_failed_rules` — which rules it failed,
   - `dq_quarantined_at` — when.
3. Assert that `count(validated) + count(quarantined) == count(input)`. Print all three.

> **Questions:**
> - A row is quarantined. Who finds out, and how? Sketch what would have to exist for this
>   to be noticed within an hour.
> - When is quarantining the wrong choice and failing the whole load the right one?

In [ ]:
# TO DO: split into validated and quarantine


# TO DO: prove nothing was lost

## Step 3 — Trend the quality

One quality check tells you about today. A quality **table** tells you that nulls in
`product_id` went from 0.1% to 4% last Tuesday, which is the sentence that actually finds
the bug.

**TO DO**

Create `dq_metrics` in your **gold** schema with one row per rule per run:

| Column | Type |
| :-- | :-- |
| `run_timestamp` | `TIMESTAMP` |
| `table_name` | `STRING` |
| `rule_name` | `STRING` |
| `rows_checked` | `BIGINT` |
| `rows_failed` | `BIGINT` |
| `failure_rate` | `DOUBLE` |

Append to it — never overwrite. Then run your checks a second time so there are two runs
in the table, and write a query showing the failure rate per rule over time.

In [ ]:
def record_dq_metrics(df: DataFrame, rules: dict, table_name: str, target: str) -> None:
    """Append one row per rule describing this run."""
    # TO DO
    pass


# TO DO: create the metrics table, record a run, record a second run, then trend it

## Step 4 — Move the rules into the table

Your code checks the rules. Nothing stops somebody bypassing the pipeline and inserting
directly.

**TO DO**

1. Add at least two `CHECK` constraints to `sales_validated` matching your rules.
2. Try an insert that violates one and read the error.
3. `SHOW TBLPROPERTIES` and find the constraints.

> **Questions:**
> - You now have the same rule in two places — the Python dict and the table constraint.
>   That is duplication. Is it the bad kind? Argue your position.
> - `ALTER TABLE ... ADD CONSTRAINT` validates existing rows. What is the deployment order
>   for adding a constraint to a table that already has violations in production?

In [ ]:
# TO DO: add constraints, attempt a bad insert, list the properties

## Step 5 — Mask what shouldn't be seen

The users dimension holds email and phone. Analysts need to join on `user_id`; they do not
need the email.

**TO DO**

Create a view `v_sales_masked` in your gold schema that joins validated sales to users and
returns the email and phone **masked**, unless the caller is in a privileged group.

```sql
CASE WHEN is_account_group_member('pii_readers') THEN email
     ELSE regexp_replace(email, '^[^@]+', '***')
END AS email
```

`is_account_group_member()` evaluates per caller, so one view serves both audiences.

**TO DO also:** query the view and confirm what *you* see. You are almost certainly not in
`pii_readers`, which is the point.

> **Questions:**
> - A view is one way. Unity Catalog also has column masks and row filters applied to the
>   table itself. What does each protect against that the other doesn't?
> - Your masking view is in gold. The unmasked table is in silver. What stops an analyst
>   from just querying silver?

In [ ]:
# TO DO: create the masked view


# TO DO: query it and check what you can see

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("B02-data-quality-and-governance")

## Recap

- One rule definition, two consumers. Duplicated predicates drift, and rows fall through
  the gap.
- Quarantine with a reason and a timestamp, or you have built a slower delete.
- Quality is a time series. A single pass/fail hides the trend that would have found the bug.
- Constraints defend the table from everything that is not your pipeline.
- `is_account_group_member()` lets one object serve callers with different entitlements.